Выгрузка данных из CDI через КХД 

Разные задачие

Загрузка набора данных в файл. Аккуратнее в файле разные ФИАС, нужно использовать корректный!!!!!!!

In [1]:

#импорт библиотек

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import oracledb
import pyodbc 
import time

pd.set_option('Display.max_columns', None)

В df данные ФИАС, это выгрузка по ИФЛ

In [2]:
#выгружаем ФИАС по домам из мастер таблицы

import oracledb
import pyodbc 
import time


import oracledb


connection = oracledb.connect(
    user = "USR_IBRAGIMOVASM",
    password = "mYJfvnul",
    dsn ="cprd-dwh-dbprim:1521/ODSPROD"
)

cursor = connection.cursor()


df = pd.read_sql_query('''select distinct "Идентификатор ФИАС дома"
                        from IFL_flat_all_062026
                       ''', connection) 

C:\Temp\Sakinat.Ibragimova\6\ipykernel_10204\1839253379.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query('''select distinct "Идентификатор ФИАС дома"


In [3]:
df.shape

(554399, 1)

In [4]:
df.columns

Index(['Идентификатор ФИАС дома'], dtype='object')

Пробуем подвязать данные по ФИАС, где это возможно

Сначала подключались к MSSQL курсор там, создаем подулючение к Оракле

In [8]:

# import oracledb


# connection = oracledb.connect(
#     user = "USR_IBRAGIMOVASM",
#     password = "mYJfvnul",
#     dsn ="cprd-dwh-dbstby:1521/ODSPROD"
# )

# cursor = connection.cursor()

In [5]:
def batch20(list_fias):
    cdi = pd.DataFrame()
    for element in list_fias:
        cursor.execute('''SELECT FEDERAL_DISTRICT,
  REGION_FIAS_ID,
  REGION_WITH_TYPE,
  AREA_FIAS_ID,
  AREA_WITH_TYPE,
  CITY_FIAS_ID,
  CITY,
  CITY_AREA,
  CITY_DISTRICT_FIAS_ID,
  CITY_DISTRICT,
  SETTLEMENT_FIAS_ID,
  SETTLEMENT,
  STREET_FIAS_ID,
  FLAT_FIAS_ID,
  HOUSE_FIAS_ID,
  FIAS_ID,
  FIAS_LEVEL,
  GEO_LAT,
  GEO_LON
               FROM DM_MOTOR.F_GET_CDI_ADDR_BY_CODE(:mybv) d''', mybv=element)
         # Получаем заголовки (имена колонок)
       # columns = [desc[0] for desc in cursor.description]
        cdi_element = cursor.fetchmany(20)
        cdi_element = pd.DataFrame(cdi_element)
        cdi = pd.concat([cdi, cdi_element])
    return cdi

In [6]:
df_cdi = pd.DataFrame()

In [7]:
data = df
count_fias = 0
list_fias = []
start_time_gl = time.time()
start_time = time.time()
for el in data['Идентификатор ФИАС дома']:
    list_fias.append(el)
    if len(list_fias) == 20:
        count_fias = count_fias + 20
        cdi_batch = batch20(list_fias)
        df_cdi = pd.concat([df_cdi, cdi_batch])
        #print((time.time() - start_time))
        if (time.time() - start_time) <= 1:
            time_sleep = 1 - (time.time() - start_time)
            print('time sleep:', time_sleep)
            time.sleep(time_sleep)
        start_time = time.time()
        list_fias = []
        if (count_fias % 50000) == 0:
            print('Обработанное количество:', count_fias)
            print("--- %s seconds ---" % (time.time() - start_time_gl))

if len(list_fias) != 0:
    cdi_batch = batch20(list_fias)
    df_cdi = pd.concat([df_cdi, cdi_batch])
print("--- %s seconds ---" % (time.time() - start_time_gl))

time sleep: 0.5623424053192139
time sleep: 0.5623421669006348
time sleep: 0.5673136711120605
time sleep: 0.5773038864135742
time sleep: 0.594003438949585
time sleep: 0.6022293567657471
time sleep: 0.6035525798797607
time sleep: 0.5737650394439697
time sleep: 0.6010503768920898
time sleep: 0.5898728370666504
time sleep: 0.5902516841888428
time sleep: 0.5635933876037598
time sleep: 0.562490701675415
time sleep: 0.6007468700408936
time sleep: 0.5802974700927734
time sleep: 0.5849242210388184
time sleep: 0.6093480587005615
time sleep: 0.5751731395721436
time sleep: 0.5930030345916748
time sleep: 0.5887792110443115
time sleep: 0.5798194408416748
time sleep: 0.5995383262634277
time sleep: 0.609398365020752
time sleep: 0.6093640327453613
time sleep: 0.6040430068969727
time sleep: 0.5971097946166992
time sleep: 0.6180250644683838
time sleep: 0.5719835758209229
time sleep: 0.5753812789916992
time sleep: 0.5953056812286377
time sleep: 0.585728645324707
time sleep: 0.590160608291626
time sleep: 0

In [8]:
len(df_cdi)

554399

In [12]:
df_cdi

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,Дальневосточный,7d468b39-1afa-41ec-8c4f-97a8603cb3d4,Хабаровский край,None,None,a4859da8-9977-4b62-8436-4e1b98c5d13f,Хабаровск,None,None,None,None,None,f7aa8e64-aed5-4d96-bb4d-21f1adad404c,None,2b900a2b-42ec-4798-b83f-a5e895250505,2b900a2b-42ec-4798-b83f-a5e895250505,8,48.3833125,135.1131713
0,Дальневосточный,b6ba5716-eb48-401b-8443-b197c9578734,Забайкальский край,None,None,2d9abaa6-85a6-4f1f-a1bd-14b76ec17d9c,Чита,None,None,None,None,None,e00dae90-5e83-4259-9529-e1a8398812ed,None,cea01844-1c3d-43fd-b5be-49370093fed5,cea01844-1c3d-43fd-b5be-49370093fed5,8,52.0791477,113.3792829
0,Приволжский,4f8b1a21-e4bb-422f-9087-d3cbf4bebc14,Пермский край,None,None,cc8b9eb5-bd4e-4472-8314-f889e8a6679c,Кизел,None,None,None,f4014092-20ae-4cdf-9bb6-44cb10896e36,Северный Коспашский,9e9079d6-61b9-4590-8377-7d03ad293cf8,None,62f1f41a-c6bf-4dc7-bfee-91d8febd3ef4,62f1f41a-c6bf-4dc7-bfee-91d8febd3ef4,8,59.0904165,57.8058123
0,Центральный,29251dcf-00a1-4e34-98d4-5c47484a36d4,Московская обл,f05b2b62-f54c-4b44-aff2-6bc3e55a640c,г Истра,05b3833e-37b3-470e-bf21-78adf1eec36f,Дедовск,None,None,None,None,None,83a2d9ee-184b-437f-82f4-2b4cb74c52f9,None,e586e7ce-f910-41dc-9cc4-132b1fe1bd08,e586e7ce-f910-41dc-9cc4-132b1fe1bd08,8,55.8658927,37.1170707
0,Южный,491cde9d-9d76-4591-ab46-ea93c079e686,респ Калмыкия,None,None,d5bd18b9-22c1-48e2-9b4a-3b7a4c89a3cb,Элиста,None,None,None,None,None,5bed2bdc-aed6-4a1f-b8c5-7dbddb7da36e,None,5c61c4bd-59a1-48b7-84b4-a072253d590a,5c61c4bd-59a1-48b7-84b4-a072253d590a,8,46.298505,44.310758
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,Сибирский,6466c988-7ce3-45e5-8b97-90ae16cb1249,Иркутская обл,None,None,82b6b7c8-82a4-44b2-8bc7-691373706b89,Ангарск,None,None,None,0af7d2f3-5af0-4f34-b42f-353cbbd9546e,12,None,None,59bc3a7b-3225-4c59-a50a-10a0d21aef0f,59bc3a7b-3225-4c59-a50a-10a0d21aef0f,8,52.5162,103.8564912
0,Центральный,a9a71961-9363-44ba-91b5-ddf0463aebc2,Тамбовская обл,None,None,88bf324d-7303-45c3-ba7f-695f2528490b,Моршанск,None,None,None,None,None,9dbe6da0-c07a-4412-b53b-11d6a46e339e,None,18056896-69aa-4959-88b8-94487acb81ef,18056896-69aa-4959-88b8-94487acb81ef,8,53.4433956,41.7723784
0,Уральский,92b30014-4d52-4e2e-892d-928142b924bf,Свердловская обл,4f51e06b-3dd4-4948-bfce-c0dc7b09f504,Сысертский р-н,None,None,None,None,None,a1d04c1e-330d-4d5a-9f0a-f20972a5d1b8,Большой Исток,67ba0b16-f265-4c2d-ac47-23ef7a6422f0,None,6b7c597e-0241-4b74-b9a9-440cc6b4f1da,6b7c597e-0241-4b74-b9a9-440cc6b4f1da,8,56.7218137,60.7802826
0,Уральский,27eb7c10-a234-44da-a59c-8b1f864966de,Челябинская обл,f530b937-a760-4270-8796-1d09daf7bbd5,Чесменский р-н,None,None,None,None,None,228b44d5-06c6-489f-a108-78634b99cb13,Новотемирский,ece63a7b-c0c8-4c40-b470-b42c2cb7eb81,None,f9b6bf83-f4c7-49e4-b0d8-0a18f081d808,f9b6bf83-f4c7-49e4-b0d8-0a18f081d808,8,53.6827269,60.1455347


In [ ]:
#df_cdi.to_csv(r'T:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\ИФЛ\2025 4 q/fias_flat_12_2025_v2.csv', encoding = 'cp1251')

In [10]:
df_cdi.to_csv(r'T:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\ИФЛ\2026 3 q/fias_flat_06_2026.csv', encoding = 'utf-8')

In [11]:
df_cdi.to_csv(r'T:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\ИФЛ\2026 3 q/fias_flat_06_2026_2.csv', encoding = 'cp1251', index=False)

In [19]:
#df_cdi.to_excel(r'T:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\ИФЛ\2025 4 q/fias_flat_12_2025_v1.xlsx')

In [87]:
df1=df.head(100)

In [ ]:
df1.to_csv(r'T:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\ИФЛ\2025 4 q/fias_flat_12_2025_v4.csv', encoding = 'cp1251', index=False)

In [ ]:
df = pd.read_excel(r'T:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\ИФЛ\2025 4 q/fias_flat_12_2025_v1.xlsx')

In [13]:
print('Всего записей по домам с уникальным ФИАС:', len(df))
print('Всего записей связалось с CDI по ФИАС:', len(df_cdi))
print('Всего записей не привязалось:', len(df)-len(df_cdi), (len(df)-len(df_cdi))/len(df)*100)

Всего записей по домам с уникальным ФИАС: 524724
Всего записей связалось с CDI по ФИАС: 524724
Всего записей не привязалось: 0 0.0


In [14]:
df.columns.to_list()

['Идентификатор ФИАС дома']

In [34]:
geo_null = len(df_cdi[(df_cdi[76].isna()==True)|(df_cdi[77].isna()==True)])
geo_null

15214

In [35]:
print('Всего записей с пустыми ГЕО данными:', geo_null, (geo_null)/len(df_cdi)*100, (geo_null)/len(df)*100)

Всего записей с пустыми ГЕО данными: 15214 2.8920215637522144 2.8920215637522144


In [79]:
df.head()

,SUGGESTION_VALUE,UNRESTRICTED_VALUE,FEDERAL_DISTRICT,REGION_FIAS_ID,REGION_WITH_TYPE,REGION_TYPE,REGION_TYPE_FULL,AREA_FIAS_ID,AREA_WITH_TYPE,AREA_TYPE,AREA_TYPE_FULL,AREA,CITY_FIAS_ID,CITY_WITH_TYPE,CITY_TYPE,CITY_TYPE_FULL,CITY,CITY_AREA,CITY_DISTRICT_FIAS_ID,CITY_DISTRICT_WITH_TYPE,CITY_DISTRICT,SETTLEMENT_FIAS_ID,SETTLEMENT_WITH_TYPE,SETTLEMENT_TYPE_FULL,SETTLEMENT,STREET_FIAS_ID,STREET_WITH_TYPE,HOUSE_FIAS_ID,HOUSE_TYPE_FULL,HOUSE,FIAS_ID,FIAS_LEVEL,CAPITAL_MARKER,OKATO,OKTMO
0,"Чувашская Республика - Чувашия, Моргаушский р-...","429533, Чувашская Республика - Чувашия, Моргау...",Приволжский,878fc621-3708-46c7-a97f-5a13a4176b3e,Чувашская Республика - Чувашия,Чувашия,Чувашия,9b1f1495-babd-4a98-ac77-0ed2c8b0aeb4,Моргаушский р-н,р-н,Моргаушский р-н,Моргаушский,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3e0ad6fd-5722-456d-860b-fedb71e06461,деревня Шатьмапоси,деревня,Шатьмапоси,b19fcf44-ec9a-4f0a-ac98-d0c80f788cab,ул Солнечная,e038f5ff-8f57-44fb-bd36-33e56ab0d6cf,домовладение,6,e038f5ff-8f57-44fb-bd36-33e56ab0d6cf,8.0,0.0,9.723287e+10,9.753200e+10
1,"г Астрахань, тер. СНТ Консервщик-3, Ореховый п...","414004, Астраханская обл, г Астрахань, тер. СН...",Южный,83009239-25cb-4561-af8e-7ee111b1cb73,Астраханская обл,обл,область,NaN,NaN,NaN,NaN,NaN,a101dd8b-3aee-4bda-9c61-9df106f145ff,г Астрахань,г,город,Астрахань,NaN,NaN,NaN,NaN,706552a5-a6d0-421d-891f-6d61676e3751,тер. СНТ Консервщик-3,территория снт,Консервщик-3,53909dde-c2ee-4b22-a70d-0c431b76de7a,Ореховый пер,79cad0c6-bb31-4a93-bb6d-438d5b1376a1,дом,19,79cad0c6-bb31-4a93-bb6d-438d5b1376a1,8.0,2.0,1.240100e+10,1.270100e+10
2,"Архангельская обл, Пинежский р-н, деревня Ерки...","164618, Архангельская обл, Пинежский р-н, дере...",Северо-Западный,294277aa-e25d-428c-95ad-46719c4ddb44,Архангельская обл,обл,область,57d43938-2c4c-43f7-a24d-e0e2fbdd478e,Пинежский р-н,р-н,Пинежский р-н,Пинежский,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,88f365f7-c184-4898-b392-7d83a6181bd7,деревня Еркино,деревня,Еркино,26d42252-9a53-4ddb-b78f-cb3fc6160044,ул Народная,b5ef596c-5ef4-4829-bb4e-f9cebd4146e8,дом,83,b5ef596c-5ef4-4829-bb4e-f9cebd4146e8,8.0,0.0,1.124882e+10,1.154800e+10
3,"Респ Башкортостан, Благовещенский р-н, село Ни...","453441, Респ Башкортостан, Благовещенский р-н,...",Приволжский,6f2cbfd8-692a-4ee4-9b16-067210bde3fc,Респ Башкортостан,Респ,республика,7f7d82dc-aa80-49a3-8502-0e3c43b35072,Благовещенский р-н,р-н,Благовещенский р-н,Благовещенский,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8b064c00-083a-4ade-89c1-a2ddfda20a3e,село Николаевка,село,Николаевка,acd78494-bde7-4916-93a1-974d4868f378,ул Верхняя,47fb6982-b660-4473-b8ae-86eada171272,дом,53,47fb6982-b660-4473-b8ae-86eada171272,8.0,0.0,8.021583e+10,8.061543e+10
4,"Кабардино-Балкарская Респ, Прохладненский р-н,...","361014, Кабардино-Балкарская Респ, Прохладненс...",Северо-Кавказский,1781f74e-be4a-4697-9c6b-493057c94818,Кабардино-Балкарская Респ,Респ,республика,c0f5c483-3a72-4244-9e18-bc6b232b92dd,Прохладненский р-н,р-н,Прохладненский р-н,Прохладненский,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,f09e0206-5470-46fd-b9a8-1cf2d331c8ba,ст-ца Приближная,станица,Приближная,be35db94-21b6-4d59-ab8b-d7bf0de65148,ул Козодерова,NaN,NaN,NaN,be35db94-21b6-4d59-ab8b-d7bf0de65148,7.0,0.0,8.322500e+10,8.362544e+07


In [78]:
df.isna().sum()

SUGGESTION_VALUE                0
UNRESTRICTED_VALUE              0
FEDERAL_DISTRICT                0
REGION_FIAS_ID                  0
REGION_WITH_TYPE                0
REGION_TYPE                     0
REGION_TYPE_FULL                0
AREA_FIAS_ID               162649
AREA_WITH_TYPE             162649
AREA_TYPE                  162649
AREA_TYPE_FULL             162649
AREA                       162649
CITY_FIAS_ID               283861
CITY_WITH_TYPE             283861
CITY_TYPE                  283861
CITY_TYPE_FULL             283861
CITY                       283861
CITY_AREA                  512703
CITY_DISTRICT_FIAS_ID      514279
CITY_DISTRICT_WITH_TYPE    473316
CITY_DISTRICT              473316
SETTLEMENT_FIAS_ID         160273
SETTLEMENT_WITH_TYPE       160273
SETTLEMENT_TYPE_FULL       160273
SETTLEMENT                 160273
STREET_FIAS_ID              65595
STREET_WITH_TYPE            65595
HOUSE_FIAS_ID               56468
HOUSE_TYPE_FULL             56468
HOUSE         

In [77]:
df = df.drop(columns=['Unnamed: 0'], errors='ignore')


In [90]:
df1.drop(columns=['GEO_LAT', 'GEO_LON'], inplace=True)

C:\Temp\Sakinat.Ibragimova\2\ipykernel_11012\726956914.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1.drop(columns=['GEO_LAT', 'GEO_LON'], inplace=True)


In [92]:
df1

,FEDERAL_DISTRICT,REGION_FIAS_ID,REGION_WITH_TYPE,REGION_TYPE,REGION_TYPE_FULL,AREA_FIAS_ID,AREA_WITH_TYPE,AREA_TYPE,AREA_TYPE_FULL,AREA,CITY_FIAS_ID,CITY_WITH_TYPE,CITY_TYPE_FULL,CITY,CITY_AREA,CITY_DISTRICT_FIAS_ID,CITY_DISTRICT_WITH_TYPE,CITY_DISTRICT,SETTLEMENT_FIAS_ID,SETTLEMENT_WITH_TYPE,SETTLEMENT_TYPE_FULL,SETTLEMENT,STREET_FIAS_ID,STREET_WITH_TYPE,HOUSE_FIAS_ID,HOUSE_TYPE_FULL,FIAS_ID,FIAS_LEVEL
0,Приволжский,878fc621-3708-46c7-a97f-5a13a4176b3e,Чувашская Республика - Чувашия,Чувашия,Чувашия,9b1f1495-babd-4a98-ac77-0ed2c8b0aeb4,Моргаушский р-н,р-н,Моргаушский р-н,Моргаушский,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3e0ad6fd-5722-456d-860b-fedb71e06461,деревня Шатьмапоси,деревня,Шатьмапоси,b19fcf44-ec9a-4f0a-ac98-d0c80f788cab,ул Солнечная,e038f5ff-8f57-44fb-bd36-33e56ab0d6cf,домовладение,e038f5ff-8f57-44fb-bd36-33e56ab0d6cf,8
1,Южный,83009239-25cb-4561-af8e-7ee111b1cb73,Астраханская обл,обл,область,NaN,NaN,NaN,NaN,NaN,a101dd8b-3aee-4bda-9c61-9df106f145ff,г Астрахань,город,Астрахань,NaN,NaN,NaN,NaN,706552a5-a6d0-421d-891f-6d61676e3751,тер. СНТ Консервщик-3,территория снт,Консервщик-3,53909dde-c2ee-4b22-a70d-0c431b76de7a,Ореховый пер,79cad0c6-bb31-4a93-bb6d-438d5b1376a1,дом,79cad0c6-bb31-4a93-bb6d-438d5b1376a1,8
2,Северо-Западный,294277aa-e25d-428c-95ad-46719c4ddb44,Архангельская обл,обл,область,57d43938-2c4c-43f7-a24d-e0e2fbdd478e,Пинежский р-н,р-н,Пинежский р-н,Пинежский,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,88f365f7-c184-4898-b392-7d83a6181bd7,деревня Еркино,деревня,Еркино,26d42252-9a53-4ddb-b78f-cb3fc6160044,ул Народная,b5ef596c-5ef4-4829-bb4e-f9cebd4146e8,дом,b5ef596c-5ef4-4829-bb4e-f9cebd4146e8,8
3,Приволжский,6f2cbfd8-692a-4ee4-9b16-067210bde3fc,Респ Башкортостан,Респ,республика,7f7d82dc-aa80-49a3-8502-0e3c43b35072,Благовещенский р-н,р-н,Благовещенский р-н,Благовещенский,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8b064c00-083a-4ade-89c1-a2ddfda20a3e,село Николаевка,село,Николаевка,acd78494-bde7-4916-93a1-974d4868f378,ул Верхняя,47fb6982-b660-4473-b8ae-86eada171272,дом,47fb6982-b660-4473-b8ae-86eada171272,8
4,Северо-Кавказский,1781f74e-be4a-4697-9c6b-493057c94818,Кабардино-Балкарская Респ,Респ,республика,c0f5c483-3a72-4244-9e18-bc6b232b92dd,Прохладненский р-н,р-н,Прохладненский р-н,Прохладненский,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,f09e0206-5470-46fd-b9a8-1cf2d331c8ba,ст-ца Приближная,станица,Приближная,be35db94-21b6-4d59-ab8b-d7bf0de65148,ул Козодерова,NaN,NaN,be35db94-21b6-4d59-ab8b-d7bf0de65148,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Уральский,92b30014-4d52-4e2e-892d-928142b924bf,Свердловская обл,обл,область,498a36b2-3311-4cbe-993e-eb706d25f8bd,Пригородный р-н,р-н,Пригородный р-н,Пригородный,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,c243a7cd-3c37-4e75-a700-a4cd838abb8e,село Лая,село,Лая,da2d0996-e73d-4782-b1d7-ba99c9969ed0,ул Ленина,a7f164aa-dd48-4366-beee-98d9d459fc07,дом,a7f164aa-dd48-4366-beee-98d9d459fc07,8
96,Северо-Западный,6d1ebb35-70c6-4129-bd55-da3969658f5d,Ленинградская обл,обл,область,cb759ab0-cd6c-4a97-8035-8052239e4551,Ломоносовский р-н,р-н,Ломоносовский р-н,Ломоносовский,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8d790f09-e748-48a5-8fe2-44ada4a1d616,деревня Гостилицы,деревня,Гостилицы,cddd958d-770d-4bac-9a46-8e2909db82e4,ул Парковая,NaN,NaN,cddd958d-770d-4bac-9a46-8e2909db82e4,7
97,Центральный,29251dcf-00a1-4e34-98d4-5c47484a36d4,Московская обл,обл,область,NaN,NaN,NaN,NaN,NaN,7c6d9ebf-8b6a-4435-a174-eb8018f7c598,г Луховицы,город,Луховицы,NaN,NaN,NaN,NaN,4bd31934-5b0b-4d87-abd1-f5f85d3bddd2,село Подлесная Слобода,село,Подлесная Слобода,NaN,NaN,137e4db1-beec-4a17-a0bd-d100ec9f9323,дом,137e4db1-beec-4a17-a0bd-d100ec9f9323,8
98,Приволжский,37a0c60a-9240-48b5-a87f-0d8c86cdb6e1,Респ Мордовия,Респ,республика,c2fb903b-6dbd-45f1-a71d-717c193a2fd4,Атяшевский р-н,р-н,Атяшевский р-н,Атяшевский,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,449118fe-deeb-42b4-a298-fe5f93d15e5a,рп Атяшево,рабочий поселок,Атяшево,d46177e9-0127-40cb-917a-fa034ec03d28,ул Мира,20ef0ac2-27f0-4a5c-8000-2eb6f65450d6,дом,

In [89]:
df1.dtypes

FEDERAL_DISTRICT            object
REGION_FIAS_ID              object
REGION_WITH_TYPE            object
REGION_TYPE                 object
REGION_TYPE_FULL            object
AREA_FIAS_ID                object
AREA_WITH_TYPE              object
AREA_TYPE                   object
AREA_TYPE_FULL              object
AREA                        object
CITY_FIAS_ID                object
CITY_WITH_TYPE              object
CITY_TYPE_FULL              object
CITY                        object
CITY_AREA                   object
CITY_DISTRICT_FIAS_ID       object
CITY_DISTRICT_WITH_TYPE     object
CITY_DISTRICT               object
SETTLEMENT_FIAS_ID          object
SETTLEMENT_WITH_TYPE        object
SETTLEMENT_TYPE_FULL        object
SETTLEMENT                  object
STREET_FIAS_ID              object
STREET_WITH_TYPE            object
HOUSE_FIAS_ID               object
HOUSE_TYPE_FULL             object
FIAS_ID                     object
FIAS_LEVEL                   int64
GEO_LAT             

In [67]:
# Удаляем строки, где FEDERAL_DISTRICT — пустое значение
df = df.dropna(subset=['FEDERAL_DISTRICT'])

In [91]:
import oracledb
import pandas as pd

# Подключение к Oracle

# Чтение CSV
#df = pd.read_excel(r'T:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\ИФЛ\2025 4 q/fias_home_09_2024_v1.xlsx')

#df = pd.read_csv(r'T:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\ИФЛ\2025 4 q/fias_home_09_2024_v3.csv', encoding = 'cp1251')

# Получаем список столбцов из DataFrame
columns = df1.columns.tolist()

# Формируем строку с именами столбцов для INSERT
columns_str = ', '.join(columns)

# Формируем строку с позиционными параметрами (:1, :2, ...)
values_str = ', '.join([f':{i+1}' for i in range(len(columns))])

# Полный SQL-запрос
insert_sql = f"""
    INSERT INTO IFL_fias_home_09_25 ({columns_str})
    VALUES ({values_str})
"""

print(insert_sql)  # Для проверки


connection = oracledb.connect(
    user = "USR_IBRAGIMOVASM",
    password = "mYJfvnul",
    dsn ="cprd-dwh-dbprim:1521/ODSPROD"
)

cursor = connection.cursor()

# Вставка данных
for index, row in df.iterrows():
    cursor.execute(insert_sql, tuple(row))

# Сохранение изменений
connection.commit()

# Закрытие соединения
cursor.close()
connection.close()


    INSERT INTO IFL_fias_home_09_25 (FEDERAL_DISTRICT, REGION_FIAS_ID, REGION_WITH_TYPE, REGION_TYPE, REGION_TYPE_FULL, AREA_FIAS_ID, AREA_WITH_TYPE, AREA_TYPE, AREA_TYPE_FULL, AREA, CITY_FIAS_ID, CITY_WITH_TYPE, CITY_TYPE_FULL, CITY, CITY_AREA, CITY_DISTRICT_FIAS_ID, CITY_DISTRICT_WITH_TYPE, CITY_DISTRICT, SETTLEMENT_FIAS_ID, SETTLEMENT_WITH_TYPE, SETTLEMENT_TYPE_FULL, SETTLEMENT, STREET_FIAS_ID, STREET_WITH_TYPE, HOUSE_FIAS_ID, HOUSE_TYPE_FULL, FIAS_ID, FIAS_LEVEL)
    VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9, :10, :11, :12, :13, :14, :15, :16, :17, :18, :19, :20, :21, :22, :23, :24, :25, :26, :27, :28)



DatabaseError: DPY-4009: 28 positional bind values are required but 30 were provided

In [102]:
import pandas as pd
from shapely.geometry import Point

In [103]:
def round_to_grid(v):
    rounded = round(v, 1)
    
    if (rounded * 10) % 2 == 0:
        if abs(v - (rounded - 0.1)) < abs(v - (rounded + 0.1)):
            return round(rounded - 0.1, 1)
        else:
            return round(rounded + 0.1, 1)
    else:
        return rounded

In [104]:
geocoded_df = pd.read_excel('Geocode_Result_2020.xlsx')
geocoded_df['longitude'] = geocoded_df['Долгота'].apply(lambda x: round_to_grid(x))
geocoded_df['latitude'] = geocoded_df['Широта'].apply(lambda x: round_to_grid(x))

FileNotFoundError: [Errno 2] No such file or directory: 'Geocode_Result_2020.xlsx'

In [ ]:


-------------------------

def round_to_grid(v):
    rounded = round(v, 1)
    
    if (rounded * 10) % 2 == 0:
        if abs(v - (rounded - 0.1)) < abs(v - (rounded + 0.1)):
            return round(rounded - 0.1, 1)
        else:
            return round(rounded + 0.1, 1)
    else:
        return rounded
----------------------
import pandas as pd
from shapely.geometry import Point

geocoded_df = pd.read_excel('Geocode_Result_2020.xlsx')
geocoded_df['longitude'] = geocoded_df['Долгота'].apply(lambda x: round_to_grid(x))
geocoded_df['latitude'] = geocoded_df['Широта'].apply(lambda x: round_to_grid(x))

-----------------------
from shapely.wkt import loads

df1 = pd.read_csv('prediction_2020.01.01_horizont_24.csv').rename(columns={'fire_proba': 'fire_prob_2020_q1'}).drop(columns=['geom'])
df2 = pd.read_csv('prediction_2020.07.01_horizont_24.csv').rename(columns={'fire_proba': 'fire_prob_2020_q2'}).drop(columns=['geom'])
df3 = pd.read_csv('prediction_2021.01.01_horizont_24.csv').rename(columns={'fire_proba': 'fire_prob_2021_q1'}).drop(columns=['geom'])
df4 = pd.read_csv('prediction_2021.07.01_horizont_24.csv').rename(columns={'fire_proba': 'fire_prob_2021_q2'}).drop(columns=['geom'])

merged = df1.merge(df2, how='outer', on=['longitude', 'latitude']) \
    .merge(df3, how='outer', on=['longitude', 'latitude']) \
    .merge(df4, how='outer', on=['longitude', 'latitude'])
-----------------------
geocoded_df = geocoded_df.merge(merged, how='left', on=['longitude', 'latitude'])
----------------------
geocoded_df.to_csv('Geocode_Result_2020+probas.csv', index=None, encoding='utf-8-sig')


Проверка объектов из КХД и CDI


In [33]:
df.head(3)

,FIAS
0,1e8bdb54-cc35-47b1-9ed7-8ba64afcc18f
1,56c6f98b-0db9-453d-826d-759241486cff
2,1c280f76-8e61-4fbe-ab47-198febf0b782


In [34]:
df_cdi.head(3)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87
0,"Рязанская обл, Касимовский р-н, село Ардабьево...","391356, Рязанская обл, Касимовский р-н, село А...",391356,Россия,RU,Центральный,963073ee-4dfc-48bd-9a70-d2dfc6bd1f31,6200000000000,RU-RYA,Рязанская обл,обл,область,None,4eee6b2e-e18d-418b-9bd6-718dec0df1c0,6200500000000,Касимовский р-н,р-н,Касимовский р-н,Касимовский,None,None,None,None,None,None,None,None,None,None,None,None,None,9a4d80e6-bb22-4e08-8710-8c0d29d75af7,6200500001300,село Ардабьево,с,село,Ардабьево,37126718-7417-4dff-8ad1-36bcbc8656f1,62005000013000400,ул Береговая,ул,улица,Береговая,1e8bdb54-cc35-47b1-9ed7-8ba64afcc18f,6200500001300040059,62:04:0170101:96,д,дом,62,None,None,None,None,None,None,None,None,None,None,None,None,None,None,1e8bdb54-cc35-47b1-9ed7-8ba64afcc18f,None,8,0,6200500001300040059,None,0,61208803001,61608403101,6226,6226,UTC+3,55.063653,41.656578,None,None,None,3,0,2,None,None,None,0
0,"Ульяновская обл, г Новоульяновск, село Криуши,...","433303, Ульяновская обл, г Новоульяновск, село...",433303,Россия,RU,Приволжский,fee76045-fe22-43a4-ad58-ad99e903bd58,7300000000000,RU-ULY,Ульяновская обл,обл,область,None,None,None,None,None,None,None,c21a50ef-de67-477c-887b-a1202730ee8e,7300000400000,г Новоульяновск,г,город,Новоульяновск,None,None,None,None,None,None,None,eb5ec8b1-70f6-4df3-ae1c-64bd87daaa7f,7300000400200,село Криуши,с,село,Криуши,4eec73dd-b6ca-4af1-85ae-c899c0723bb2,73000004002000900,ул Карла Маркса,ул,улица,Карла Маркса,56c6f98b-0db9-453d-826d-759241486cff,7300000400200090037,73:19:112202:335,двлд,домовладение,27,None,None,None,None,None,None,None,None,None,None,None,None,None,None,56c6f98b-0db9-453d-826d-759241486cff,None,8,0,7300000400200090037,517766,0,73415000002,73715000106,7300,7300,UTC+4,54.107073,48.534322,None,None,None,2,0,2,None,None,None,0
0,"Калининградская обл, г Светлый, поселок Взморь...","238345, Калининградская обл, г Светлый, посело...",238345,Россия,RU,Северо-Западный,90c7181e-724f-41b3-b6c6-bd3ec7ae3f30,3900000000000,RU-KGD,Калининградская обл,обл,область,None,None,None,None,None,None,None,6d379259-b8d5-4cca-9412-1ded0849fd1a,3900000600000,г Светлый,г,город,Светлый,None,None,None,None,None,None,None,223d596b-d46d-4f75-b9e4-d37e9cad2883,3900000600600,поселок Взморье,п,поселок,Взморье,26c30c5a-c1c6-45a6-b25b-065cd7648266,39000006006002000,ул Советская,ул,улица,Советская,1c280f76-8e61-4fbe-ab47-198febf0b782,3900000600600200016,39:18:010023:318,д,дом,21,None,None,None,None,None,None,None,None,None,None,33.0,None,None,None,1c280f76-8e61-4fbe-ab47-198febf0b782,None,8,0,3900000600600200016,6930743,0,27425000009,27725000116,3900,3900,UTC+2,54.697994,20.250569,None,None,None,2,0,2,None,None,None,0


In [135]:
connection = oracledb.connect(
    user = "BaevPP",
    password = "njsbK13SowoSk#",
    dsn ="cprd-dwh-dbstby:1521/ODSPROD"
)

cursor = connection.cursor()

In [140]:

cursor.execute(u'''select m_all."Страхователь Адрес", m_all."ССО Адрес объекта страхования", 
                m_all."1С Адрес объекта страхования", m_all."ФИАС Адрес", m_all."Идентификатор ФИАС"
 
  from P_BI_1C_DBO.ZUD_ANLSEGM_MODULEPRODUCT_V m_all
  inner join BaevPP.FIAS_MODULEPRODUCT_V m
  ON m_all."Идентификатор ФИАС" = m.FIAS''')

#for column in cursor.description:
#    print(column)
q=cursor.fetchmany(10000)

In [136]:

cursor.execute(u'''select m_all."Страхователь Адрес", m_all."ССО Адрес объекта страхования", 
                m_all."1С Адрес объекта страхования", m_all."ФИАС Адрес", m_all."Идентификатор ФИАС"
 
  from P_BI_1C_DBO.ZUD_ANLSEGM_HOMEPROTECTION_V m_all
  inner join BaevPP.FIAS_HOMEPROTECTION_V m
  ON m_all."Идентификатор ФИАС" = m.FIAS''')

#for column in cursor.description:
#    print(column)
q=cursor.fetchmany(10000)

In [137]:
data_homepro = pd.DataFrame(q)

In [138]:
data_homepro.head(3)

,0,1,2,3,4
0,"682890, Хабаровский край, Ванинский р-н, Окт...",None,"682890,27,Р-Н ВАНИНСКИЙ,П ОКТЯБРЬСКИЙ,УЛ ЮЖНАЯ...","682890, Россия, край Хабаровский, р-н Ванински...",7c8be9e7-ee2b-41aa-a364-dc3f8561e9e7
1,"125167, Москва г, Москва г, Красноармейская...",None,"125167,Москва г,Красноармейская ул,д.2,корп.1,...","125167, Россия, г Москва, г Москва, ул Красноа...",760861d7-36f2-4f40-af99-cff53cc635ef
2,"603032, Нижегородская обл, Нижний Новгород г,...",None,"603032,НИЖЕГОРОДСКАЯ ОБЛ,НИЖНИЙ НОВГОРОД Г,КОС...","603032, Россия, обл Нижегородская, г Нижний Но...",f445e929-18d1-40b6-9b49-26c941a29915


In [141]:
data = pd.DataFrame(q)

In [80]:
df_cdi.head(3)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87
0,"Рязанская обл, Касимовский р-н, село Ардабьево...","391356, Рязанская обл, Касимовский р-н, село А...",391356,Россия,RU,Центральный,963073ee-4dfc-48bd-9a70-d2dfc6bd1f31,6200000000000,RU-RYA,Рязанская обл,обл,область,None,4eee6b2e-e18d-418b-9bd6-718dec0df1c0,6200500000000,Касимовский р-н,р-н,Касимовский р-н,Касимовский,None,None,None,None,None,None,None,None,None,None,None,None,None,9a4d80e6-bb22-4e08-8710-8c0d29d75af7,6200500001300,село Ардабьево,с,село,Ардабьево,37126718-7417-4dff-8ad1-36bcbc8656f1,62005000013000400,ул Береговая,ул,улица,Береговая,1e8bdb54-cc35-47b1-9ed7-8ba64afcc18f,6200500001300040059,62:04:0170101:96,д,дом,62,None,None,None,None,None,None,None,None,None,None,None,None,None,None,1e8bdb54-cc35-47b1-9ed7-8ba64afcc18f,None,8,0,6200500001300040059,None,0,61208803001,61608403101,6226,6226,UTC+3,55.063653,41.656578,None,None,None,3,0,2,None,None,None,0
0,"Ульяновская обл, г Новоульяновск, село Криуши,...","433303, Ульяновская обл, г Новоульяновск, село...",433303,Россия,RU,Приволжский,fee76045-fe22-43a4-ad58-ad99e903bd58,7300000000000,RU-ULY,Ульяновская обл,обл,область,None,None,None,None,None,None,None,c21a50ef-de67-477c-887b-a1202730ee8e,7300000400000,г Новоульяновск,г,город,Новоульяновск,None,None,None,None,None,None,None,eb5ec8b1-70f6-4df3-ae1c-64bd87daaa7f,7300000400200,село Криуши,с,село,Криуши,4eec73dd-b6ca-4af1-85ae-c899c0723bb2,73000004002000900,ул Карла Маркса,ул,улица,Карла Маркса,56c6f98b-0db9-453d-826d-759241486cff,7300000400200090037,73:19:112202:335,двлд,домовладение,27,None,None,None,None,None,None,None,None,None,None,None,None,None,None,56c6f98b-0db9-453d-826d-759241486cff,None,8,0,7300000400200090037,517766,0,73415000002,73715000106,7300,7300,UTC+4,54.107073,48.534322,None,None,None,2,0,2,None,None,None,0
0,"Калининградская обл, г Светлый, поселок Взморь...","238345, Калининградская обл, г Светлый, посело...",238345,Россия,RU,Северо-Западный,90c7181e-724f-41b3-b6c6-bd3ec7ae3f30,3900000000000,RU-KGD,Калининградская обл,обл,область,None,None,None,None,None,None,None,6d379259-b8d5-4cca-9412-1ded0849fd1a,3900000600000,г Светлый,г,город,Светлый,None,None,None,None,None,None,None,223d596b-d46d-4f75-b9e4-d37e9cad2883,3900000600600,поселок Взморье,п,поселок,Взморье,26c30c5a-c1c6-45a6-b25b-065cd7648266,39000006006002000,ул Советская,ул,улица,Советская,1c280f76-8e61-4fbe-ab47-198febf0b782,3900000600600200016,39:18:010023:318,д,дом,21,None,None,None,None,None,None,None,None,None,None,33.0,None,None,None,1c280f76-8e61-4fbe-ab47-198febf0b782,None,8,0,3900000600600200016,6930743,0,27425000009,27725000116,3900,3900,UTC+2,54.697994,20.250569,None,None,None,2,0,2,None,None,None,0


In [142]:
data_cdi = df_cdi[[1, 5, 9, 43, 44, 76, 77]]
data_cdi

,1,5,9,43,44,76,77
0,"391356, Рязанская обл, Касимовский р-н, село А...",Центральный,Рязанская обл,Береговая,1e8bdb54-cc35-47b1-9ed7-8ba64afcc18f,55.063653,41.656578
0,"433303, Ульяновская обл, г Новоульяновск, село...",Приволжский,Ульяновская обл,Карла Маркса,56c6f98b-0db9-453d-826d-759241486cff,54.107073,48.534322
0,"238345, Калининградская обл, г Светлый, посело...",Северо-Западный,Калининградская обл,Советская,1c280f76-8e61-4fbe-ab47-198febf0b782,54.697994,20.250569
0,"693005, Сахалинская обл, г Южно-Сахалинск, тер...",Дальневосточный,Сахалинская обл,Западная,None,46.959179,142.738041
0,"623795, Свердловская обл, Артемовский р-н, сел...",Уральский,Свердловская обл,Максима Горького,dec40ca9-32fe-43f1-abbc-3fdd4fbfecab,57.360303,61.696428
...,...,...,...,...,...,...,...
0,"425207, Респ Марий Эл, Медведевский р-н, дерев...",Приволжский,Респ Марий Эл,Центральная,c1614c0a-fab5-4b10-ae93-64a3c68bf526,56.809738,47.293012
0,"617470, Пермский край, г Кунгур, ул Луговая, д 40",Приволжский,Пермский край,Луговая,3186e0d0-78df-40a4-a96f-ade6b78fb594,57.434285,56.917912
0,"644007, Омская обл, г Омск, Центральный округ,...",Сибирский,Омская обл,Герцена,b156d317-8e91-49de-bf23-7e8c6a76370d,55.006167,73.373503
0,"607676, Нижегородская обл, Кстовский р-н, дере...",Приволжский,Нижегородская обл,Мира,565e9776-2c26-42e9-af65-4ee2115fa473,56.117226,44.235935


In [143]:
data_cdi.columns = ['1', '5', '9', '43', 'fias', '76', '77']
data.columns = ['0', '1', '2', '3', 'fias']
data_homepro.columns = ['0', '1', '2', '3', 'fias']

In [144]:
data = data.merge(data_cdi, how='left', left_on = 'fias', right_on = 'fias')
data_homepro = data_homepro.merge(data_cdi, how='left', left_on = 'fias', right_on = 'fias')

In [87]:
data[data['5'].isna()==False]

,0,1_x,2,3,fias,1_y,5,9,43,76,77
2,"353901, Краснодарский край, Новороссийск г, ...","Краснодарский край, НОВОРОССИЙСК, БАЙКАЛЬСКАЯ,...","Краснодарский край, НОВОРОССИЙСК, БАЙКАЛЬСКАЯ,...","353901, Россия, край Краснодарский, г Новоросс...",43a57cf2-ba1a-4b0d-a072-6b0f72ad041a,"353901, Краснодарский край, г Новороссийск, ул...",Южный,Краснодарский край,Байкальская,44.7482444,37.7880273
3,"347750, Ростовская обл, Зерноградский р-н, М...","Ростовская обл, ЗЕРНОГРАДСКИЙ, МЕЧЕТИНСКАЯ, ОК...","Ростовская обл, ЗЕРНОГРАДСКИЙ, МЕЧЕТИНСКАЯ, ОК...","347750, Россия, обл Ростовская, р-н Зерноградс...",afc7e192-2bc9-46cb-8ac2-5dbcc20a2dbb,"347750, Ростовская обл, Зерноградский р-н, ст-...",Южный,Ростовская обл,Октябрьская,46.768075,40.461719
8,"623815, Свердловская обл, Ирбитский р-н, Рет...","Свердловская обл, ирбитский, ретнева, демина, 2-1",None,"623815, Россия, обл Свердловская, р-н Ирбитски...",9e645b16-58d2-4511-a2c3-12921f6dee3d,"623815, Свердловская обл, Ирбитский р-н, дерев...",Уральский,Свердловская обл,Демина,57.5522,62.670292
12,"628241, Ханты-Мансийский Автономный округ - Юг...","Ханты-Мансийский Автономный округ - Югра АО, С...","Ханты-Мансийский Автономный округ - Югра АО, С...","628241, Россия, АО Ханты-Мансийский Автономный...",272a49f8-1220-4162-824c-c199dc817d11,"628241, Ханты-Мансийский Автономный округ - Юг...",Уральский,Ханты-Мансийский Автономный округ - Югра,Свердлова,61.385403,63.568195
15,"612020, Кировская обл, Шабалинский р-н, Лени...","Кировская обл, ШАБАЛИНСКИЙ, ЛЕНИНСКОЕ, ШКОЛЬНА...","Кировская обл, ШАБАЛИНСКИЙ, ЛЕНИНСКОЕ, ШКОЛЬНА...","612020, Россия, обл Кировская, р-н Шабалинский...",26e6459f-7f66-4851-bc5d-0cc2b4ee8ab5,"612020, Кировская обл, Шабалинский р-н, пгт Ле...",Приволжский,Кировская обл,Школьная,58.31017,47.0812
...,...,...,...,...,...,...,...,...,...,...,...
9985,"347942, Ростовская обл, Таганрог г, 1-й Нов...","Ростовская обл, Таганрог, Василенко, 53","Ростовская обл, Таганрог, Василенко, 53","347909, Россия, обл Ростовская, г Таганрог, ул...",889acaa6-5dd3-4462-90ad-d66c083c70ab,"347909, Ростовская обл, г Таганрог, ул Василен...",Южный,Ростовская обл,Василенко,47.26564,38.938931
9987,"453800, Башкортостан Респ, Хайбуллинский р-н, ...","Башкортостан Респ, Хайбуллинский, Акъяр, Ахмет...","Башкортостан Респ, Хайбуллинский, Акъяр, Ахмет...","453800, Россия, Респ Башкортостан, р-н Хайбулл...",fa604e28-bfb5-4524-b4b8-507127f75ccf,"453800, Респ Башкортостан, Хайбуллинский р-н, ...",Приволжский,Респ Башкортостан,Ахметшина,51.862382,58.207093
9991,"394088, Воронежская обл, Воронеж г, Победы ...","Воронежская обл, ВОРОНЕЖ, ПОБЕДЫ, д 43, кв. 263",None,"394088, Россия, обл Воронежская, г Воронеж, б-...",9348607c-a87f-4a8d-9a47-92d284c8f948,"394088, Воронежская обл, г Воронеж, б-р Победы...",Центральный,Воронежская обл,Победы,51.711334,39.145004
9994,"404131, Волгоградская обл, Волжский г, Мира...","Волгоградская обл, ВОЛЖСКИЙ, МИРА, д 79, кв. 270",None,"404131, Россия, обл Волгоградская, г Волжский,...",8e4bcae0-4f2a-48e6-8289-b3913384fe7a,"404131, Волгоградская обл, г Волжский, ул Мира...",Южный,Волгоградская обл,Мира,48.768,44.80846


In [88]:
data[data['5'].isna()==False].to_excel('ifl_sverka.xlsx', encoding = 'utf-8')

In [145]:
data_homepro[data_homepro['5'].isna()==False].to_excel('ifl_sverka_homepro.xlsx', encoding = 'utf-8')

Связка по объектов и квадратов со скорингом

In [96]:
df_predictions = pd.read_csv(r'T:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\ИФЛ\3 кв 2023\Данные\fires_predictions.csv', encoding = 'utf-8', delimiter=';')

In [155]:
data_homepro[data_homepro['5'].isna()==False].head(100)

,0,1_x,2,3,fias,1_y,5,9,43,76,77
77,"399000, Липецкая обл, Измалковский р-н, Изма...",None,липецкая обл измалковский р-он с измалково ул ...,"399000, Россия, обл Липецкая, р-н Измалковский...",ab06520e-6a92-470f-a9ab-4cebca2d8d72,"399000, Липецкая обл, Измалковский р-н, село И...",Центральный,Липецкая обл,Пушкина,52.692177,37.951164
104,"450005, Башкортостан Респ, Уфа г, Революцио...",None,"РФ,Республика Башкортостан,Уфимский район,с/с ...","450515, Россия, Респ Башкортостан, р-н Уфимски...",efed52d9-9ca1-4365-aeda-e48ae3b49d73,"450515, Респ Башкортостан, Уфимский р-н, дерев...",Приволжский,Респ Башкортостан,None,54.868446,55.732316
125,"174571, Новгородская обл, Хвойнинский р-н, Ю...",None,НОВГОРОДСКАЯ ОБЛ.П.ЮБИЛЕЙНЫЙ УЛ.СОЛНЕЧНАЯ Д.4 ...,"174571, Россия, обл Новгородская, р-н Хвойнинс...",ba6fc36f-effa-4406-9ba7-8481b58f26aa,"174571, Новгородская обл, Хвойнинский р-н, пос...",Северо-Западный,Новгородская обл,Солнечная,58.837809,35.002029
126,"416502, Астраханская обл, Ахтубинский р-н, Ахт...","с.Капустин Яр,ул.Гоголя.,д.15",None,"416510, Россия, обл Астраханская, р-н Ахтубинс...",6dc0340c-ad96-4754-9d6b-93804fb6f5f0,"416510, Астраханская обл, Ахтубинский р-н, сел...",Южный,Астраханская обл,Гоголя,48.577274,45.752094
268,"603138, Нижегородская обл, Нижний Новгород г,...",None,"606324,НИЖЕГОРОДСКАЯ ОБЛ,ДАЛЬНЕКОНСТАНТИНОВСКИ...","606310, Россия, обл Нижегородская, р-н Дальнек...",e8b29a41-794a-48e1-a4da-7d182669dc16,"606310, Нижегородская обл, Дальнеконстантиновс...",Приволжский,Нижегородская обл,None,55.755221,44.23141
...,...,...,...,...,...,...,...,...,...,...,...
1758,"429540, Чувашская Республика - Чувашия, Моргау...",None,чувашия моргаушский р-н д.сидуккасы ул.сидуков...,"429540, Россия, Чувашия Чувашская Республика -...",9d2ae718-697c-49c3-9aaa-2409c6ba36a1,"429540, Чувашская республика - Чувашия, Моргау...",Приволжский,Чувашская республика - Чувашия,Сидуковская,56.044754,46.772528
1792,"184372, Мурманская обл, Видяево п, Централь...",МО Сергиево-Посадский р-н с.Мишутино д.344,None,"141337, Россия, обл Московская, г Сергиев Поса...",1f13c19f-6efb-4c17-ad7e-988ac14410b1,"141337, Московская обл, г Сергиев Посад, село ...",Центральный,Московская обл,None,56.384864,38.101171
1800,"150517, Ярославская обл, Ярославский р-н, Ми...","Ярославская обл., Ярославский р-н, п. Михайлов...",None,"150517, Россия, обл Ярославская, р-н Ярославск...",e3990ceb-c351-46c1-bc5b-78c9c92f46ba,"150517, Ярославская обл, Ярославский р-н, посе...",Центральный,Ярославская обл,Южная,57.781428,39.738612
1843,"450112, Башкортостан Респ, Уфа г, Шумавцова...",None,"52405,БАШКОРТОСТАН РЕСП,ИГЛИНСКИЙ Р-Н,,УРМАН С...","452405, Россия, Респ Башкортостан, р-н Иглинск...",ba1814ac-1bc8-4ef1-9b67-e4258f1adf98,"452405, Респ Башкортостан, Иглинский р-н, село...",Приволжский,Респ Башкортостан,Ленина,54.883307,56.877704


In [156]:
data_homepro[data_homepro['5']=='Дальневосточный']

,0,1_x,2,3,fias,1_y,5,9,43,76,77
627,"670018, Бурятия Респ, Иволгинский р-н, Посел...",None,РБ ИВОЛГИНСКИЙ Р- Н С ПОСЕЛЬЕ УЛ ЗАОВРАЖНАЯ Д5,"670018, Россия, Респ Бурятия, р-н Иволгинский,...",d3a79960-6f07-4865-9465-b3681736eb62,"670018, Респ Бурятия, Иволгинский р-н, село По...",Дальневосточный,Респ Бурятия,Заовражная,51.805715,107.540593
1301,"678600, Саха /Якутия/ Респ, Амгинский у, Амг...","РС(Я),Амгинский р-он,с.Амга ул.Матросова,д.10",None,"678600, Россия, Респ Саха /Якутия/, у Амгински...",aaddc180-5078-4fd2-aa76-bdf1e0886d53,"678600, Респ Саха /Якутия/, Амгинский улус, се...",Дальневосточный,Респ Саха /Якутия/,Матросова,60.900648,131.976807
1638,"672038, Забайкальский край, Чита г, Простор...",None,"672038,ЗАБАЙКАЛЬСКИЙ КРАЙ,ЧИТА Г,ПРОСТОРНАЯ УЛ...","672038, Россия, край Забайкальский, г Чита, ул...",8ea4a5bd-21b1-4196-9ded-6ef3a67e4f61,"672038, Забайкальский край, г Чита, ул Простор...",Дальневосточный,Забайкальский край,Просторная,52.097665,113.477468
1938,"670045, Бурятия Респ, Улан-Удэ г, Амбулатор...",рб г Улан-Удэ ул амбулаторная д 29,None,"670045, Россия, Респ Бурятия, г Улан-Удэ, ул А...",df9c9d27-9da2-40ef-8850-680b8b06e142,"670045, Респ Бурятия, г Улан-Удэ, Железнодорож...",Дальневосточный,Респ Бурятия,Амбулаторная,51.828314,107.625331
2319,"690037, Приморский край, Владивосток г, Лад...",None,"692549,ПРИМОРСКИЙ КРАЙ,МИХАЙЛОВСКИЙ Р-Н,ОТРАДН...","692549, Россия, край Приморский, р-н Михайловс...",40ae1abf-bf9a-49e7-96ed-d4886f648896,"692549, Приморский край, Михайловский р-н, сел...",Дальневосточный,Приморский край,Садовая,43.813524,132.533502
2435,"676290, Амурская обл, Тында г, Ташкентская ...",None,"676290,АМУРСКАЯ ОБЛ,ТЫНДА Г,ТАШКЕНТСКАЯ УЛ,Д.4...","676290, Россия, обл Амурская, г Тында, ул Ташк...",dc9eb34c-1a6e-453a-b8b3-df77718f2fd7,"676290, Амурская обл, г Тында, ул Ташкентская,...",Дальневосточный,Амурская обл,Ташкентская,55.1657223,124.7101013
2588,None,None,Г.ХАБАРОВСК УЛ.ЯКУТСКАЯ Д.4,"680003, Россия, край Хабаровский, г Хабаровск,...",42370944-571c-4ddd-bb82-da928507ab23,"680003, Хабаровский край, г Хабаровск, ул Якут...",Дальневосточный,Хабаровский край,Якутская,48.39085,135.09366
2742,"671635, Бурятия Респ, Курумканский р-н, Арзг...",None,"670000,БУРЯТИЯ РЕСП,ИВОЛГИНСКИЙ Р-Н,ПОСЕЛЬЕ СН...","670018, Россия, Респ Бурятия, р-н Иволгинский,...",5d030d60-9648-49a1-bf7d-b57632b83454,"670018, Респ Бурятия, Иволгинский р-н, село По...",Дальневосточный,Респ Бурятия,Метельная,51.81306,107.551049
3392,"670045, Бурятия Респ, Улан-Удэ г, Амбулатор...",рб г Улан-Удэ ул амбулаторная д 29,None,"670045, Россия, Респ Бурятия, г Улан-Удэ, ул А...",df9c9d27-9da2-40ef-8850-680b8b06e142,"670045, Респ Бурятия, г Улан-Удэ, Железнодорож...",Дальневосточный,Респ Бурятия,Амбулаторная,51.828314,107.625331
3488,"676380, Амурская обл, Серышевский р-н, Бочка...",None,"676380,АМУРСКАЯ ОБЛ,СЕРЫШЕВСКИЙ Р-Н,БОЧКАРЕВКА...","676380, Россия, обл Амурская, р-н Серышевский,...",bf018012-4b8c-41e5-97f6-c6d5955b87c6,"676380, Амурская обл, Серышевский р-н, село Бо...",Дальневосточный,Амурская обл,Заводская,50.958826,128.4473


In [153]:
data.loc[9633]['3']

'677901, Россия, Респ Саха /Якутия/, г Якутск, ул Экспериментальная (мкр Марха), дом 18'

In [97]:
df_predictions

,longitude,latitude,fire_proba,geom
0,19.9,54.5,0.204553,"POLYGON ((19.799999999999997 54.4, 19.79999999..."
1,19.9,54.7,0.032333,"POLYGON ((19.799999999999997 54.6, 19.79999999..."
2,20.1,54.3,0.003595,"POLYGON ((20 54.199999999999996, 20 54.4, 20.2..."
3,20.1,54.5,0.450748,"POLYGON ((20 54.4, 20 54.6, 20.200000000000003..."
4,20.1,54.7,0.063083,"POLYGON ((20 54.6, 20 54.800000000000004, 20.2..."
...,...,...,...,...
69595,168.9,68.1,0.001553,"POLYGON ((168.8 68, 168.8 68.19999999999999, 1..."
69596,168.9,68.3,0.000240,"POLYGON ((168.8 68.2, 168.8 68.39999999999999,..."
69597,168.9,68.5,0.000367,"POLYGON ((168.8 68.4, 168.8 68.6, 169 68.6, 16..."
69598,168.9,68.7,0.000225,"POLYGON ((168.8 68.60000000000001, 168.8 68.8,..."


In [98]:
df_cdi

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87
0,"Рязанская обл, Касимовский р-н, село Ардабьево...","391356, Рязанская обл, Касимовский р-н, село А...",391356,Россия,RU,Центральный,963073ee-4dfc-48bd-9a70-d2dfc6bd1f31,6200000000000,RU-RYA,Рязанская обл,обл,область,None,4eee6b2e-e18d-418b-9bd6-718dec0df1c0,6200500000000,Касимовский р-н,р-н,Касимовский р-н,Касимовский,None,None,None,None,None,None,None,None,None,None,None,None,None,9a4d80e6-bb22-4e08-8710-8c0d29d75af7,6200500001300,село Ардабьево,с,село,Ардабьево,37126718-7417-4dff-8ad1-36bcbc8656f1,62005000013000400,ул Береговая,ул,улица,Береговая,1e8bdb54-cc35-47b1-9ed7-8ba64afcc18f,6200500001300040059,62:04:0170101:96,д,дом,62,None,None,None,None,None,None,None,None,None,None,None,None,None,None,1e8bdb54-cc35-47b1-9ed7-8ba64afcc18f,None,8,0,6200500001300040059,None,0,61208803001,61608403101,6226,6226,UTC+3,55.063653,41.656578,None,None,None,3,0,2,None,None,None,0
0,"Ульяновская обл, г Новоульяновск, село Криуши,...","433303, Ульяновская обл, г Новоульяновск, село...",433303,Россия,RU,Приволжский,fee76045-fe22-43a4-ad58-ad99e903bd58,7300000000000,RU-ULY,Ульяновская обл,обл,область,None,None,None,None,None,None,None,c21a50ef-de67-477c-887b-a1202730ee8e,7300000400000,г Новоульяновск,г,город,Новоульяновск,None,None,None,None,None,None,None,eb5ec8b1-70f6-4df3-ae1c-64bd87daaa7f,7300000400200,село Криуши,с,село,Криуши,4eec73dd-b6ca-4af1-85ae-c899c0723bb2,73000004002000900,ул Карла Маркса,ул,улица,Карла Маркса,56c6f98b-0db9-453d-826d-759241486cff,7300000400200090037,73:19:112202:335,двлд,домовладение,27,None,None,None,None,None,None,None,None,None,None,None,None,None,None,56c6f98b-0db9-453d-826d-759241486cff,None,8,0,7300000400200090037,517766,0,73415000002,73715000106,7300,7300,UTC+4,54.107073,48.534322,None,None,None,2,0,2,None,None,None,0
0,"Калининградская обл, г Светлый, поселок Взморь...","238345, Калининградская обл, г Светлый, посело...",238345,Россия,RU,Северо-Западный,90c7181e-724f-41b3-b6c6-bd3ec7ae3f30,3900000000000,RU-KGD,Калининградская обл,обл,область,None,None,None,None,None,None,None,6d379259-b8d5-4cca-9412-1ded0849fd1a,3900000600000,г Светлый,г,город,Светлый,None,None,None,None,None,None,None,223d596b-d46d-4f75-b9e4-d37e9cad2883,3900000600600,поселок Взморье,п,поселок,Взморье,26c30c5a-c1c6-45a6-b25b-065cd7648266,39000006006002000,ул Советская,ул,улица,Советская,1c280f76-8e61-4fbe-ab47-198febf0b782,3900000600600200016,39:18:010023:318,д,дом,21,None,None,None,None,None,None,None,None,None,None,33.0,None,None,None,1c280f76-8e61-4fbe-ab47-198febf0b782,None,8,0,3900000600600200016,6930743,0,27425000009,27725000116,3900,3900,UTC+2,54.697994,20.250569,None,None,None,2,0,2,None,None,None,0
0,"г Южно-Сахалинск, тер. СНТ Общепитовец-1, ул З...","693005, Сахалинская обл, г Южно-Сахалинск, тер...",693005,Россия,RU,Дальневосточный,aea6280f-4648-460f-b8be-c2bc18923191,6500000000000,RU-SAK,Сахалинская обл,обл,область,None,None,None,None,None,None,None,44388ad0-06aa-49b0-bbf9-1704629d1d68,6500000100000,г Южно-Сахалинск,г,город,Южно-Сахалинск,None,None,None,None,None,None,None,b2bd9a1b-0d84-4393-8105-7c82033db1f6,65000001000099500,тер. СНТ Общепитовец-1,тер. СНТ,территория снт,Общепитовец-1,23fa336f-ca27-445a-92f7-97ee0b7366b2,65000001000099800,ул Западная,ул,улица,Западная,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,23fa336f-ca27-445a-92f7-97ee0b7366b2,None,7,0,65000001000099800,2119441,2,64401000000,64701000001,6500,6500,UTC+11,46.959179,142.738041,None,None,None,4,4,10,None,None,None,0
0,"Свердловская обл, Артемовский р-н, село Покров...","623795, Свердловская обл, Артемовский р-н, сел...",623795,Россия,RU,Уральский,92b30014-4d52-4e2e-892d-928142b924bf,6600000000000,RU-SVE,Свердловская обл,обл,область,None,3c101200-191d

In [102]:
from shapely.geometry import Point
def round_to_grid(v):
    rounded = round(v, 1)
    
    if (rounded * 10) % 2 == 0:
        if abs(v - (rounded - 0.1)) < abs(v - (rounded + 0.1)):
            return round(rounded - 0.1, 1)
        else:
            return round(rounded + 0.1, 1)
    else:
        return rounded

In [106]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 334228 entries, 0 to 0
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   1          334228 non-null  object 
 1   5          333421 non-null  object 
 2   9          334228 non-null  object 
 3   43         292579 non-null  object 
 4   fias       295773 non-null  object 
 5   latitude   329658 non-null  float64
 6   longitude  329658 non-null  float64
dtypes: float64(2), object(5)
memory usage: 20.4+ MB


In [115]:
data = df_cdi[[1, 5, 9, 43, 44, 76, 77]]
data.columns = ['1', '5', '9', '43', 'fias', 'latitude', 'longitude']

In [116]:
data['longitude'] = data['longitude'].astype('float')
data['latitude'] = data['latitude'].astype('float')

C:\Temp\Pavel.Baev\ipykernel_11268\1653182492.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['longitude'] = data['longitude'].astype('float')
C:\Temp\Pavel.Baev\ipykernel_11268\1653182492.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['latitude'] = data['latitude'].astype('float')


In [117]:
data['longitude_c'] = data['longitude'].apply(lambda x: round_to_grid(x))
data['latitude_c'] = data['latitude'].apply(lambda x: round_to_grid(x))

C:\Temp\Pavel.Baev\ipykernel_11268\2494061645.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['longitude_c'] = data['longitude'].apply(lambda x: round_to_grid(x))
C:\Temp\Pavel.Baev\ipykernel_11268\2494061645.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['latitude_c'] = data['latitude'].apply(lambda x: round_to_grid(x))


In [118]:
data

,1,5,9,43,fias,latitude,longitude,longitude_c,latitude_c
0,"391356, Рязанская обл, Касимовский р-н, село А...",Центральный,Рязанская обл,Береговая,1e8bdb54-cc35-47b1-9ed7-8ba64afcc18f,55.063653,41.656578,41.7,55.1
0,"433303, Ульяновская обл, г Новоульяновск, село...",Приволжский,Ульяновская обл,Карла Маркса,56c6f98b-0db9-453d-826d-759241486cff,54.107073,48.534322,48.5,54.1
0,"238345, Калининградская обл, г Светлый, посело...",Северо-Западный,Калининградская обл,Советская,1c280f76-8e61-4fbe-ab47-198febf0b782,54.697994,20.250569,20.3,54.7
0,"693005, Сахалинская обл, г Южно-Сахалинск, тер...",Дальневосточный,Сахалинская обл,Западная,None,46.959179,142.738041,142.7,46.9
0,"623795, Свердловская обл, Артемовский р-н, сел...",Уральский,Свердловская обл,Максима Горького,dec40ca9-32fe-43f1-abbc-3fdd4fbfecab,57.360303,61.696428,61.7,57.3
...,...,...,...,...,...,...,...,...,...
0,"425207, Респ Марий Эл, Медведевский р-н, дерев...",Приволжский,Респ Марий Эл,Центральная,c1614c0a-fab5-4b10-ae93-64a3c68bf526,56.809738,47.293012,47.3,56.9
0,"617470, Пермский край, г Кунгур, ул Луговая, д 40",Приволжский,Пермский край,Луговая,3186e0d0-78df-40a4-a96f-ade6b78fb594,57.434285,56.917912,56.9,57.5
0,"644007, Омская обл, г Омск, Центральный округ,...",Сибирский,Омская обл,Герцена,b156d317-8e91-49de-bf23-7e8c6a76370d,55.006167,73.373503,73.3,55.1
0,"607676, Нижегородская обл, Кстовский р-н, дере...",Приволжский,Нижегородская обл,Мира,565e9776-2c26-42e9-af65-4ee2115fa473,56.117226,44.235935,44.3,56.1


In [119]:
data = data.merge(df_predictions, how='left', left_on=['longitude_c', 'latitude_c'], right_on=['longitude', 'latitude'])

In [121]:
display(data[data['fire_proba'].isna()==False])
display(data[data['fire_proba'].isna()==True])

,1,5,9,43,fias,latitude_x,longitude_x,longitude_c,latitude_c,longitude_y,latitude_y,fire_proba,geom
0,"391356, Рязанская обл, Касимовский р-н, село А...",Центральный,Рязанская обл,Береговая,1e8bdb54-cc35-47b1-9ed7-8ba64afcc18f,55.063653,41.656578,41.7,55.1,41.7,55.1,0.012220,"POLYGON ((41.6 55, 41.6 55.2, 41.8000000000000..."
1,"433303, Ульяновская обл, г Новоульяновск, село...",Приволжский,Ульяновская обл,Карла Маркса,56c6f98b-0db9-453d-826d-759241486cff,54.107073,48.534322,48.5,54.1,48.5,54.1,0.079348,"POLYGON ((48.4 54, 48.4 54.2, 48.6 54.2, 48.6 ..."
2,"238345, Калининградская обл, г Светлый, посело...",Северо-Западный,Калининградская обл,Советская,1c280f76-8e61-4fbe-ab47-198febf0b782,54.697994,20.250569,20.3,54.7,20.3,54.7,0.837505,"POLYGON ((20.2 54.6, 20.2 54.800000000000004, ..."
3,"693005, Сахалинская обл, г Южно-Сахалинск, тер...",Дальневосточный,Сахалинская обл,Западная,None,46.959179,142.738041,142.7,46.9,142.7,46.9,0.227585,"POLYGON ((142.6 46.8, 142.6 47, 142.7999999999..."
4,"623795, Свердловская обл, Артемовский р-н, сел...",Уральский,Свердловская обл,Максима Горького,dec40ca9-32fe-43f1-abbc-3fdd4fbfecab,57.360303,61.696428,61.7,57.3,61.7,57.3,0.016347,"POLYGON ((61.6 57.199999999999996, 61.6 57.4, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
334223,"425207, Респ Марий Эл, Медведевский р-н, дерев...",Приволжский,Респ Марий Эл,Центральная,c1614c0a-fab5-4b10-ae93-64a3c68bf526,56.809738,47.293012,47.3,56.9,47.3,56.9,0.007844,"POLYGON ((47.199999999999996 56.8, 47.19999999..."
334224,"617470, Пермский край, г Кунгур, ул Луговая, д 40",Приволжский,Пермский край,Луговая,3186e0d0-78df-40a4-a96f-ade6b78fb594,57.434285,56.917912,56.9,57.5,56.9,57.5,0.020016,"POLYGON ((56.8 57.4, 56.8 57.6, 57 57.6, 57 57..."
334225,"644007, Омская обл, г Омск, Центральный округ,...",Сибирский,Омская обл,Герцена,b156d317-8e91-49de-bf23-7e8c6a76370d,55.006167,73.373503,73.3,55.1,73.3,55.1,0.326839,"POLYGON ((73.2 55, 73.2 55.2, 73.3999999999999..."
334226,"607676, Нижегородская обл, Кстовский р-н, дере...",Приволжский,Нижегородская обл,Мира,565e9776-2c26-42e9-af65-4ee2115fa473,56.117226,44.235935,44.3,56.1,44.3,56.1,0.028770,"POLYGON ((44.199999999999996 56, 44.1999999999..."


,1,5,9,43,fias,latitude_x,longitude_x,longitude_c,latitude_c,longitude_y,latitude_y,fire_proba,geom
157,"367006, Респ Дагестан, г Махачкала, Кировский ...",Северо-Кавказский,Респ Дагестан,Пирамидальная,None,42.981629,47.409146,47.5,42.9,NaN,NaN,NaN,NaN
206,"346940, Ростовская обл, Куйбышевский р-н, село...",Южный,Ростовская обл,Лесной,2aa1c451-3efd-48a8-8392-3ecf6f2cf30b,47.807700,38.915036,38.9,47.9,NaN,NaN,NaN,NaN
262,"Самарская обл, Красноярский р-н, тер. СНТ Мета...",Приволжский,Самарская обл,6-я,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
276,"238561, Калининградская обл, Светлогорский р-н...",Северо-Западный,Калининградская обл,12-я линия,None,54.935617,20.095484,20.1,54.9,NaN,NaN,NaN,NaN
380,"368304, Респ Дагестан, г Каспийск, Колос СНТ, ...",Северо-Кавказский,Респ Дагестан,Линия 3,None,42.891586,47.636671,47.7,42.9,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
334015,"186757, Респ Карелия, г Сортавала, пгт Вяртсил...",Северо-Западный,Респ Карелия,Мира,7cef7c8f-4b23-4412-84d9-9e626465595b,62.174564,30.692127,30.7,62.1,NaN,NaN,NaN,NaN
334077,"452429, Респ Башкортостан, Иглинский р-н, тер ...",Приволжский,Респ Башкортостан,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
334080,"367031, Респ Дагестан, г Махачкала, Ленинский ...",Северо-Кавказский,Респ Дагестан,Достоевского,None,42.948589,47.521480,47.5,42.9,NaN,NaN,NaN,NaN
334100,"368607, Респ Дагестан, г Дербент, ул Пилотная,...",Северо-Кавказский,Респ Дагестан,Пилотная,9ed3f4cc-b900-438a-bac8-1bc7107e06b0,42.099750,48.277134,48.3,42.1,NaN,NaN,NaN,NaN


In [122]:
data['latitude_c'].unique()

array([55.1, 54.1, 54.7, 46.9, 57.3, 55.3, 56.9, 45.3, 52.1, 56.7, 44.5,
       48.7, 53.7, 54.9, 62.5, 44.7, 45.1, 55.9, 44.9, 56.1, 53.1, 59.7,
       47.3, 43.3, 55.5, 43.7, 56.5, 51.9, 49.7, 55.7, 61.7, 51.7, 53.3,
       58.1, 53.5, 50.5, 43.9, 53.9, 58.3, 45.7, 44.3, 58.5, 57.9, 54.3,
       54.5, 51.1, 50.7, 44.1, 41.7, 46.5, 56.3, 58.9, 61.3, 51.5, 57.1,
       46.3, 43.1, 52.7, 43.5, 57.5, 51.3, 47.1, 57.7, 50.3, 49.1, 58.7,
       52.5, 42.9, 49.9, 50.1, 52.9, 47.5, 47.7, 52.3, 45.5, 47.9, 42.7,
       46.7, 60.7, 59.1, 61.1, 59.3,  nan, 42.1, 60.9, 61.5, 59.5, 49.5,
       45.9, 59.9, 66.1, 42.5, 50.9, 65.1, 61.9, 46.1, 60.1, 62.7, 48.5,
       41.9, 63.1, 64.7, 60.3, 62.1, 48.9, 64.5, 63.9, 62.3, 48.1, 48.3,
       62.9, 63.5, 49.3, 64.3, 42.3, 63.7, 63.3, 60.5, 68.9, 67.7, 66.5,
       64.1, 65.5, 67.1, 64.9, 65.9, 41.5, 66.3, 41.3, 69.1, 65.3, 67.5,
       68.7, 67.3, 65.7, 71.9, 67.9, 68.3, 66.7, 68.5, 70.1, 66.9, 68.1,
       69.3, 70.7, 69.9, 70.9, 71.7, 69.5, 72.9, 73

In [128]:
data[data['fire_proba'].isna()==True]['latitude_c'].unique()

array([42.9, 47.9,  nan, 54.9, 47.7, 42.5, 53.1, 46.7, 45.5, 45.3, 43.5,
       50.1, 61.5, 42.7, 46.1, 42.1, 51.7, 47.5, 46.9, 64.1, 47.1, 46.5,
       49.1, 55.1, 57.9, 48.3, 44.5, 50.3, 43.3, 51.3, 59.3, 50.9, 42.3,
       52.3, 44.7, 43.9, 53.7, 48.9, 60.5, 61.1, 49.9, 54.1, 59.7, 49.5,
       45.1, 64.7, 58.9, 60.9, 57.7, 60.7, 41.5, 66.7, 61.7, 41.9, 56.7,
       44.3, 45.7, 64.5, 50.5, 62.1, 43.1, 51.1, 64.3, 57.1, 59.9, 50.7,
       53.9, 69.1, 67.7, 44.9, 61.9, 59.1, 66.1, 68.7, 51.5, 59.5, 49.3,
       65.1, 65.9, 53.5, 69.3, 47.3, 56.9, 55.3, 54.5, 45.9, 53.3, 66.3,
       52.7, 69.5, 44.1, 67.9, 54.7, 48.1, 65.7])

In [131]:
df_predictions.query('latitude == 41.9')

,longitude,latitude,fire_proba,geom
8293,46.1,41.9,0.032990,"POLYGON ((46 41.8, 46 42, 46.2 42, 46.2 41.8, ..."
8420,46.3,41.9,0.055160,"POLYGON ((46.199999999999996 41.8, 46.19999999..."
8547,46.5,41.9,0.117387,"POLYGON ((46.4 41.8, 46.4 42, 46.6 42, 46.6 41..."
8672,46.7,41.9,0.024833,"POLYGON ((46.6 41.8, 46.6 42, 46.8000000000000..."
8795,46.9,41.9,0.018477,"POLYGON ((46.8 41.8, 46.8 42, 47 42, 47 41.8, ..."
8913,47.1,41.9,0.058179,"POLYGON ((47 41.8, 47 42, 47.2 42, 47.2 41.8, ..."
9025,47.3,41.9,0.454261,"POLYGON ((47.199999999999996 41.8, 47.19999999..."
9134,47.5,41.9,0.845898,"POLYGON ((47.4 41.8, 47.4 42, 47.6 42, 47.6 41..."
9240,47.7,41.9,0.906894,"POLYGON ((47.6 41.8, 47.6 42, 47.8000000000000..."
9340,47.9,41.9,0.854012,"POLYGON ((47.8 41.8, 47.8 42, 48 42, 48 41.8, ..."


In [132]:
data[data['fire_proba'].isna()==True].query('latitude_c == 41.9')

,1,5,9,43,fias,latitude_x,longitude_x,longitude_c,latitude_c,longitude_y,latitude_y,fire_proba,geom
30381,"368193, Респ Дагестан, Курахский р-н, село Кумук",Северо-Кавказский,Респ Дагестан,None,None,41.835312,48.418953,48.5,41.9,NaN,NaN,NaN,NaN
54448,"368796, Респ Дагестан, Магарамкентский р-н, се...",Северо-Кавказский,Респ Дагестан,Хаджи-Давуда,a6bb69e6-ba90-427f-a0f7-14f0089761d3,41.810823,48.509807,48.5,41.9,NaN,NaN,NaN,NaN
56418,"368615, Респ Дагестан, Дербентский р-н, село Н...",Северо-Кавказский,Респ Дагестан,None,41666795-4991-4c7e-890f-c4826e6aa1dd,41.860047,48.435292,48.5,41.9,NaN,NaN,NaN,NaN
73971,"368615, Респ Дагестан, Дербентский р-н, село Н...",Северо-Кавказский,Респ Дагестан,Школьная,c040d14a-f4a6-4052-8c18-80ff704df279,41.860047,48.435292,48.5,41.9,NaN,NaN,NaN,NaN
84320,"368795, Респ Дагестан, Магарамкентский р-н, се...",Северо-Кавказский,Респ Дагестан,Дачная,a0ad4da4-ea7a-4c14-b849-48ac6ce7fb0f,41.823820,48.483091,48.5,41.9,NaN,NaN,NaN,NaN
87342,"368796, Респ Дагестан, Магарамкентский р-н, се...",Северо-Кавказский,Респ Дагестан,Победы,f25ef18f-315f-4c45-ace5-1a624a08a0fc,41.818693,48.542865,48.5,41.9,NaN,NaN,NaN,NaN
102622,"368615, Респ Дагестан, Дербентский р-н, село Н...",Северо-Кавказский,Респ Дагестан,Речная,4a078d6f-ae31-4107-8731-0e80b14ade75,41.860047,48.435292,48.5,41.9,NaN,NaN,NaN,NaN
109447,"368796, Респ Дагестан, Магарамкентский р-н, се...",Северо-Кавказский,Респ Дагестан,None,None,41.808135,48.522374,48.5,41.9,NaN,NaN,NaN,NaN
112423,"368615, Респ Дагестан, Дербентский р-н, село Б...",Северо-Кавказский,Респ Дагестан,Шоссейная,da16544c-686f-4ebf-ab50-7187e0f1a5de,41.890890,48.413382,48.5,41.9,NaN,NaN,NaN,NaN
132591,"368796, Респ Дагестан, Магарамкентский р-н, се...",Северо-Кавказский,Респ Дагестан,Лезгинцева,c18a9aa4-b76c-42a2-ab5b-b5c95288cdfe,41.810823,48.509807,48.5,41.9,NaN,NaN,NaN,NaN
